In [1]:
import pandas as pd
import warnings
import numpy as np
import seaborn as sns
warnings.filterwarnings('ignore')

# Games

<html>
    You need to know:
<ol>
<li>We need to remove the questions that were answered more than 1 time. Repeated replies. Maintain only the 1st attempt.</li>
<li>Questions that were not replied on time. Empty replies, we look at the effect.</li>
<li>Remove questions that were replied less than X, X=5.</li>
<li>Remove users that replied less than X, X=5.</li>
<li>Classify bios by country and whether they are alive or dead.</li>
<li>Compute average fraction of correct replies for each group.</li>
</ol>
</html>

In [2]:
# data = pd.read_csv("../../data_popularity/shared/year_month_data_082023.csv", engine='python')

In [3]:
games = pd.read_csv("data/data_pantheon_trivia_games.csv", engine='python')
games.loc[games['question_famous_person']=='Al-Baydawi','question_deathyear'] = 1286
games.loc[games['question_famous_person'].str.contains('Bernardo_OHiggins'), 'question_birthyear'] = 1778
games.loc[games['question_famous_person'].str.contains('Bernardo_OHiggins'), 'question_deathyear'] = 1842
# q_id is the order of the question
# question_id is the id of the question
games.head()

,user_id,person_country,person_age,person_gender,question_id,q_id,question_famous_person,question_bplace_country,question_birthyear,question_deathyear,question_occupation,question_correct_answer,question_hpi,question_gender,question_date,question_time,current_answer,correct_answer_option,question_type,correct
0,3040ae1a-cb34-4749-97a3-a07ffba9697c,United States,30-39,Men,14,1,Thomas Edison,United States,1847.0,1931.0,inventor,inventor,92.594708,M,2022-11-16,19:50:49.283,Inventor,b,medium,1
1,3040ae1a-cb34-4749-97a3-a07ffba9697c,United States,30-39,Men,14,5,Thomas Edison,United States,1847.0,1931.0,inventor,inventor,92.594708,M,2022-11-16,19:55:04.995,Basketball player,a,medium,0
2,3040ae1a-cb34-4749-97a3-a07ffba9697c,United States,30-39,Men,14,8,Thomas Edison,United States,1847.0,1931.0,inventor,inventor,92.594708,M,2022-11-16,19:57:26.862,Journalist,a,medium,0
3,3a23013b-3eaf-4100-8943-03dd61909052,NaN,NaN,NaN,14,2,Thomas Edison,United States,1847.0,1931.0,inventor,inventor,92.594708,M,2022-11-16,05:40:39.149,Inventor,c,medium,1
4,1c0b00b4-2e62-4376-9a07-f3b2073e5090,NaN,NaN,NaN,14,5,Thomas Edison,United States,1847.0,1931.0,inventor,inventor,92.594708,M,2022-11-16,06:15:46.200,Inventor,c,medium,1


In [7]:
games[games['question_date']<'2022-11-04']

,user_id,person_country,person_age,person_gender,question_id,q_id,question_famous_person,question_bplace_country,question_birthyear,question_deathyear,question_occupation,question_correct_answer,question_hpi,question_gender,question_date,question_time,current_answer,correct_answer_option,question_type,correct
20694,6116f604-885b-40eb-a184-7cbe0add0fcb,Germany,30-39,Men,59,1,Frank Sinatra,United States,1915.0,1998.0,singer,Frank Sinatra,82.152871,M,2022-11-02,06:20:32.632,Frank Sinatra,b,hard,1
20695,6116f604-885b-40eb-a184-7cbe0add0fcb,Germany,30-39,Men,59,1,Frank Sinatra,United States,1915.0,1998.0,singer,Frank Sinatra,82.152871,M,2022-11-02,06:24:54.683,Frank Sinatra,b,hard,1
20696,64671f9a-8a5e-4bd5-9ab7-62083020d096,NaN,NaN,NaN,59,3,Frank Sinatra,United States,1915.0,1998.0,singer,Frank Sinatra,82.152871,M,2022-11-02,06:27:02.207,Frank Sinatra,d,hard,1
20697,64671f9a-8a5e-4bd5-9ab7-62083020d096,NaN,NaN,NaN,59,7,Frank Sinatra,United States,1915.0,1998.0,singer,Frank Sinatra,82.152871,M,2022-11-02,06:29:17.668,Frank Sinatra,c,hard,1
20698,e25e2773-2455-439d-a65a-ce1bf2c2c4bb,NaN,NaN,NaN,59,8,Frank Sinatra,United States,1915.0,1998.0,singer,Frank Sinatra,82.152871,M,2022-11-02,08:28:20.608,,d,hard,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24428,b49cad7c-2208-4a56-805b-18936c84d99a,NaN,NaN,NaN,2962,4,Al Taliaferro,United States,1905.0,1969.0,comic artist,United States,63.248620,M,2022-03-04,16:18:23.544,Uzbekistan,b,easy,0
24429,b49cad7c-2208-4a56-805b-18936c84d99a,NaN,NaN,NaN,2964,7,Jon Cryer,United States,1965.0,NaN,actor,actress/actor,63.676400,M,2022-03-04,16:19:09.360,Geologist,d,medium,0
24430,031f95a7-934a-4114-a638-9372e49facc7,NaN,NaN,NaN,2964,4,Jon Cryer,United States,1965.0,NaN,actor,actress/actor,63.676400,M,2022-03-04,16:23:44.625,Actress/actor,d,medium,1
24431,b49cad7c-2208-4a56-805b-18936c84d99a,NaN,NaN,NaN,2966,8,Laura Antonelli,Croatia,1941.0,2015.0,actor,actress/actor,73.315250,F,2022-03-04,16:19:17.975,Mathematician,b,medium,0


In [5]:
games['question_date'].min(), games['question_date'].max()

('2022-03-04', '2024-04-12')

In [5]:
aux = games[['user_id','person_country']].drop_duplicates()

aux[aux['person_country'].isna()]

,user_id,person_country
3,3a23013b-3eaf-4100-8943-03dd61909052,NaN
4,1c0b00b4-2e62-4376-9a07-f3b2073e5090,NaN
5,5b2e6950-790c-409d-9c3c-9db8f6de3ad3,NaN
15,fcbaa4e0-555f-4f95-8815-85210338d3c3,NaN
16,15733dcf-0eb4-48f6-b72b-fa1169ba39b2,NaN
...,...,...
414206,4474f2df-5177-4b2b-a3bd-163b9313971b,NaN
414220,98d54440-878a-4e6e-ac64-6bb3eb6f1619,NaN
414240,ee29bf8f-60b3-4555-98c8-3bddf680c461,NaN
414250,e6a825c4-0e1e-4d49-8ad2-aaf68007ba90,NaN


In [4]:
games['user_id'].nunique()

41070

In [5]:
games['user_id'].count()

414290

In [6]:
print("Empty replies: ")
len(games[games['current_answer']==' '])

Empty replies: 


20995

In [7]:
print("Non-empty replies: ")
len(games[games['current_answer']!=' '])

Non-empty replies: 


393295

In [30]:
len(games[games['current_answer']!=' '])/len(games)

0.949322938038572

In [31]:
games['question_famous_person'].nunique()

1581

In [32]:
games[['user_id','person_country']].drop_duplicates()['person_country'].value_counts().head()

United States     3600
United Kingdom     728
Canada             570
Germany            413
Australia          347
Name: person_country, dtype: int64

In [33]:
games[['question_famous_person','question_bplace_country']].drop_duplicates()['question_bplace_country'].value_counts().head()

United States     438
France            281
United Kingdom    218
Italy             174
Germany           108
Name: question_bplace_country, dtype: int64

In [34]:
games['question_date'].min(),games['question_date'].max()

('2022-03-04', '2024-04-12')

In [35]:
people_country = games[['user_id','person_country']].drop_duplicates().reset_index()
"People we do not know the country: ", people_country[people_country['person_country'].isna()]['user_id'].nunique(), people_country[people_country['person_country'].isna()]['user_id'].nunique()/people_country['user_id'].nunique()

('People we do not know the country: ', 31232, 0.7604577550523497)

In [36]:
"People that we analyse based on country: ", people_country[~people_country['person_country'].isna()]['user_id'].nunique(), people_country[~people_country['person_country'].isna()]['user_id'].nunique()/people_country['user_id'].nunique()

('People that we analyse based on country: ', 9838, 0.23954224494765036)

In [37]:
print("% replies without country: ",games[games['person_country'].isna()]['question_id'].count()/games['question_id'].count(), 
"\n % replies with country: ", games[~games['person_country'].isna()]['question_id'].count()/games['question_id'].count())

% replies without country:  0.6894614883294311 
 % replies with country:  0.31053851167056895


In [38]:
"Number of replies:",len(games), "Number of biographies:", games['question_famous_person'].nunique(),"Number of unique questions:", games['question_id'].nunique()

('Number of replies:',
 414290,
 'Number of biographies:',
 1581,
 'Number of unique questions:',
 1837)

In [39]:
"Dead people: ", games[(games['question_birthyear']<2015) & 
             (games['question_deathyear']<2015)]['question_famous_person'].nunique(), "N of questions: ",\
                games[(games['question_birthyear']<2015) & 
             (games['question_deathyear']<2015)]['question_id'].nunique(),"N of replies: ",\
                games[(games['question_birthyear']<2015) & 
             (games['question_deathyear']<2015)]['question_id'].count()

('Dead people: ', 1132, 'N of questions: ', 1325, 'N of replies: ', 313866)

In [40]:
dead_people = games[(games['question_birthyear']<2015) & 
             (games['question_deathyear']<2015)]['question_famous_person'].unique()
print(len(dead_people))

1132


In [41]:
"Alive people: ", games[ (games['question_birthyear']>=1915) &  
             (games['question_deathyear'].isna())]['question_famous_person'].nunique(), "N of questions: ",\
                games[ (games['question_birthyear']>=1915) &  
             (games['question_deathyear'].isna())]['question_id'].nunique(), "N of replies: ",\
                games[ (games['question_birthyear']>=1915) &  
             (games['question_deathyear'].isna())]['question_id'].count()

('Alive people: ', 386, 'N of questions: ', 446, 'N of replies: ', 88978)

In [42]:
alive_people = games[ (games['question_birthyear']>=1915) &  
             (games['question_deathyear'].isna())]['question_famous_person'].unique()
print(len(alive_people))

386


In [43]:
people_included = np.hstack([games[(games['question_birthyear']<2015) & 
             (games['question_deathyear']<2015)]['question_famous_person'].unique(),
          games[ (games['question_birthyear']>=1915) &  
             (games['question_deathyear'].isna())]['question_famous_person'].unique()])
len(people_included)

1518

In [44]:
"People that we are not including: ", games[~games['question_famous_person'].isin(people_included)]['question_famous_person'].nunique()

('People that we are not including: ', 63)

In [45]:
"Lack of data: ", games[(games['question_birthyear']<=1915) & (games['question_deathyear'].isna())]['question_famous_person'].unique()

('Lack of data: ', array([], dtype=object))

In [46]:
"Died between 2015-2024: ", games[~games['question_famous_person'].isin(people_included) &\
            (~games['question_deathyear'].isna())]['question_famous_person'].unique()

('Died between 2015-2024: ',
 array(['Stephen Hawking', 'Alan Rickman', 'Sean Connery', 'Dario Fo',
        'Bernardo Bertolucci', 'Debbie Reynolds', 'Günter Grass',
        'Christopher Lee', 'Bud Spencer', 'Terry Pratchett',
        'David Rockefeller', 'David J. Thouless', 'Iolanda Balaș',
        'Károly Palotai', 'Isao Tomita', 'Pietro Anastasi',
        'Mahasweta Devi', 'John Havlicek', 'David Koch', 'Doris Roberts',
        'James Ingram', "Maureen O'Hara", 'Neville Marriner',
        'Danny Aiello', 'Isao Takahata', 'Zito', 'William Peter Blatty',
        'Laura Antonelli', 'Yevgeny Yevtushenko', 'Hans Erni', 'Bill Nunn',
        'Robert M. Pirsig', 'Yoichiro Nambu', 'Sergio Sollima',
        'Penny Marshall', 'Sydney Brenner', 'Ben Cross',
        'Jean Starobinski', 'Jiří Menzel', 'E. L. Doctorow',
        'John Ashbery', 'Raymond Kopa', 'Ennio Morricone',
        'Cynthia Lennon', 'Roy J. Glauber', 'Lucia Bosè', 'Luke Perry',
        'Jacque Fresco', 'Jacques Rivette', 'Buc

In [47]:
games['country_comparison'] = None
games.loc[games['question_bplace_country']==games['person_country'],'country_comparison'] = 'Same'
games.loc[games['question_bplace_country']!=games['person_country'],'country_comparison'] = 'Different'
games.loc[games['person_country'].isna(),'country_comparison'] = None
games['country_comparison'].value_counts()

Different    109903
Same          18750
Name: country_comparison, dtype: int64

In [48]:
games['alive_dead'] = None
games.loc[games['question_famous_person'].isin(alive_people),'alive_dead'] = 'Alive'
games.loc[games['question_famous_person'].isin(dead_people),'alive_dead'] = 'Dead'
games.loc[games['question_famous_person'].isna(),'alive_dead'] = None
games['alive_dead'].value_counts()

Dead     313866
Alive     88978
Name: alive_dead, dtype: int64

In [49]:
"All data: ", games['correct'].mean(), games[games['current_answer']!=' ']['correct'].mean()

('All data: ', 0.42472905452702214, 0.4474020773210948)

In [50]:
"Separate based on the country: ", games.groupby('country_comparison')['correct'].mean(), \
            games[games['current_answer']!=' '].groupby('country_comparison')['correct'].mean()

('Separate based on the country: ',
 country_comparison
 Different    0.465392
 Same         0.490080
 Name: correct, dtype: float64,
 country_comparison
 Different    0.492926
 Same         0.510500
 Name: correct, dtype: float64)

In [51]:
games['us_sample'] = None
games.loc[games['person_country']=="United States",'us_sample'] = 'Americans'
games.loc[games['person_country']!="United States",'us_sample'] = 'Non-Americans'
games.loc[games['person_country'].isna(),'us_sample'] = None
games['us_sample'].value_counts()

Non-Americans    83100
Americans        45553
Name: us_sample, dtype: int64

In [52]:
"Separate based on USA: ", games.groupby('country_comparison')['correct'].mean(), \
            games[games['current_answer']!=' '].groupby('country_comparison')['correct'].mean()

('Separate based on USA: ',
 country_comparison
 Different    0.465392
 Same         0.490080
 Name: correct, dtype: float64,
 country_comparison
 Different    0.492926
 Same         0.510500
 Name: correct, dtype: float64)

In [53]:
"Repeated replies, people replied more than one time a question",
# Add code here to get the first question replied for the duplicates - check if it is correct
# we can create with attempts (1st, 2nd, 3rd)
nonrepeated_data = games.sort_values(by=['question_date','question_time'], ascending=True)\
            .drop_duplicates(subset=['question_id','user_id'],keep='first')
"Number of replies:",len(nonrepeated_data), "Number of biographies:", nonrepeated_data['question_famous_person'].nunique(),"Number of unique questions:", nonrepeated_data['question_id'].nunique()


('Number of replies:',
 81482,
 'Number of biographies:',
 1581,
 'Number of unique questions:',
 1837)

In [54]:
"Dead people: ", nonrepeated_data[(nonrepeated_data['question_birthyear']<2015) & 
             (nonrepeated_data['question_deathyear']<2015)]['question_famous_person'].nunique(), "N of questions: ",\
                nonrepeated_data[(nonrepeated_data['question_birthyear']<2015) & 
             (nonrepeated_data['question_deathyear']<2015)]['question_id'].nunique(),"N of replies: ",\
                nonrepeated_data[(nonrepeated_data['question_birthyear']<2015) & 
             (nonrepeated_data['question_deathyear']<2015)]['question_id'].count()

('Dead people: ', 1132, 'N of questions: ', 1325, 'N of replies: ', 62488)

In [55]:
"Alive people: ", nonrepeated_data[ (nonrepeated_data['question_birthyear']>=1915) &  
             (nonrepeated_data['question_deathyear'].isna())]['question_famous_person'].nunique(), "N of questions: ",\
                nonrepeated_data[ (games['question_birthyear']>=1915) &  
             (nonrepeated_data['question_deathyear'].isna())]['question_id'].nunique(), "N of replies: ",\
                nonrepeated_data[ (games['question_birthyear']>=1915) &  
             (nonrepeated_data['question_deathyear'].isna())]['question_id'].count()

('Alive people: ', 386, 'N of questions: ', 446, 'N of replies: ', 16444)

In [56]:
nonrepeated_data['country_comparison'].value_counts()

Different    22878
Same          3257
Name: country_comparison, dtype: int64

In [57]:
nonrepeated_data['alive_dead'].value_counts()

Dead     62488
Alive    16444
Name: alive_dead, dtype: int64

In [58]:
"All data: ", nonrepeated_data['correct'].mean(), nonrepeated_data[nonrepeated_data['current_answer']!=' ']['correct'].mean()

('All data: ', 0.445067622296949, 0.46742885130954837)

In [59]:
"Separate based on the country: ", nonrepeated_data.groupby('country_comparison')['correct'].mean(), \
            nonrepeated_data[nonrepeated_data['current_answer']!=' '].groupby('country_comparison')['correct'].mean()

('Separate based on the country: ',
 country_comparison
 Different    0.501442
 Same         0.560025
 Name: correct, dtype: float64,
 country_comparison
 Different    0.526553
 Same         0.580522
 Name: correct, dtype: float64)

In [60]:
nonrepeated_data.groupby(['alive_dead','country_comparison'])['correct'].mean().reset_index().pivot_table(index='country_comparison', columns='alive_dead', values='correct')

alive_dead,Alive,Dead
country_comparison,,
Different,0.502965,0.501931
Same,0.552434,0.564806


In [61]:
nonrepeated_data.groupby(['alive_dead','country_comparison'])['correct'].sem().reset_index().pivot_table(index='country_comparison', columns='alive_dead', values='correct')

alive_dead,Alive,Dead
country_comparison,,
Different,0.008029,0.003688
Same,0.015223,0.011072


In [62]:
nonrepeated_data[nonrepeated_data['current_answer']!=' '].groupby(['alive_dead','country_comparison'])['correct'].mean().reset_index().pivot_table(index='country_comparison', columns='alive_dead', values='correct')

alive_dead,Alive,Dead
country_comparison,,
Different,0.522216,0.528287
Same,0.570600,0.587351


In [63]:
nonrepeated_data[nonrepeated_data['current_answer']!=' '].groupby(['alive_dead','country_comparison'])['correct'].sem().reset_index().pivot_table(index='country_comparison', columns='alive_dead', values='correct')

alive_dead,Alive,Dead
country_comparison,,
Different,0.008173,0.003778
Same,0.015401,0.011212


In [64]:
print("Based on slug")
remove_questions = nonrepeated_data.groupby('question_famous_person')['correct'].count().reset_index()
remove_questions = remove_questions[remove_questions['correct']<5]['question_famous_person'].unique()
print("removed", len(remove_questions), nonrepeated_data[(nonrepeated_data['question_famous_person'].isin(remove_questions))]['question_famous_person'].count())

Based on slug
removed 367 1127


In [65]:
print("Based on questions")
remove_questions = nonrepeated_data.groupby('question_id')['correct'].count().reset_index()
remove_questions = remove_questions[remove_questions['correct']<5]['question_id'].unique()
print("removed", len(remove_questions), nonrepeated_data[(nonrepeated_data['question_id'].isin(remove_questions))]['question_id'].count())

Based on questions
removed 412 1271


In [66]:
print("Based on people")
remove_people = nonrepeated_data.groupby('user_id')['correct'].count().reset_index()
remove_people = remove_people[remove_people['correct']<5]['user_id'].unique()
print("removed", len(remove_people), nonrepeated_data[(nonrepeated_data['user_id'].isin(remove_people))]['question_id'].count())

Based on people
removed 37155 39192


In [67]:
remove_questions = nonrepeated_data.groupby('question_id')['correct'].count().reset_index()
remove_questions = remove_questions[remove_questions['correct']<5]['question_id'].unique()

remove_people = nonrepeated_data.groupby('user_id')['correct'].count().reset_index()
remove_people = remove_people[remove_people['correct']<5]['user_id'].unique()

print("removed", len(remove_questions),len(remove_people))

nonrepeated_data = nonrepeated_data[(~nonrepeated_data['question_id'].isin(remove_questions))] 
nonrepeated_data = nonrepeated_data[(~nonrepeated_data['user_id'].isin(remove_people))] 
len(nonrepeated_data)

removed 412 37155


41231

In [68]:
nonrepeated_data.groupby(['alive_dead','country_comparison'])['correct'].mean().reset_index()\
        .pivot_table(index='country_comparison', columns='alive_dead', values='correct')

alive_dead,Alive,Dead
country_comparison,,
Different,0.560645,0.536559
Same,0.670807,0.649919


In [69]:
nonrepeated_data[nonrepeated_data['current_answer']!=' '].groupby(['alive_dead','country_comparison'])['correct'].mean().reset_index()\
        .pivot_table(index='country_comparison', columns='alive_dead', values='correct')

alive_dead,Alive,Dead
country_comparison,,
Different,0.583407,0.569461
Same,0.695279,0.680815


In [70]:
nonrepeated_data[nonrepeated_data['current_answer']!=' '].groupby(['alive_dead','country_comparison'])['correct'].sem().reset_index()\
        .pivot_table(index='country_comparison', columns='alive_dead', values='correct')

alive_dead,Alive,Dead
country_comparison,,
Different,0.010359,0.004608
Same,0.021345,0.013588


In [71]:
print("Number of replies")
nonrepeated_data[nonrepeated_data['current_answer']!=' '].groupby(['alive_dead','country_comparison'])['correct'].count().reset_index()\
        .pivot_table(index='country_comparison', columns='alive_dead', values='correct')

Number of replies


alive_dead,Alive,Dead
country_comparison,,
Different,2266,11546
Same,466,1178


In [72]:
print("Number of bios")
nonrepeated_data[nonrepeated_data['current_answer']!=' '].groupby(['alive_dead','country_comparison'])['question_famous_person'].nunique().reset_index()\
        .pivot_table(index='country_comparison', columns='alive_dead', values='question_famous_person')

Number of bios


alive_dead,Alive,Dead
country_comparison,,
Different,233,799
Same,120,274


In [73]:
nonrepeated_data.groupby(['alive_dead','us_sample'])['correct'].mean().reset_index()\
        .pivot_table(index='us_sample', columns='alive_dead', values='correct')

alive_dead,Alive,Dead
us_sample,,
Americans,0.634286,0.558761
Non-Americans,0.554934,0.540645


In [74]:
nonrepeated_data[nonrepeated_data['current_answer']!=' '].groupby(['alive_dead','us_sample'])['correct'].mean().reset_index()\
        .pivot_table(index='us_sample', columns='alive_dead', values='correct')

alive_dead,Alive,Dead
us_sample,,
Americans,0.660714,0.597168
Non-Americans,0.576638,0.570641


In [75]:
print("Number of replies")
nonrepeated_data[nonrepeated_data['current_answer']!=' '].groupby(['alive_dead','us_sample'])['correct'].count().reset_index()\
        .pivot_table(index='us_sample', columns='alive_dead', values='correct')

Number of replies


alive_dead,Alive,Dead
us_sample,,
Americans,840,4379
Non-Americans,1892,8345


In [76]:
print("Number of bios")
nonrepeated_data[nonrepeated_data['current_answer']!=' '].groupby(['alive_dead','us_sample'])['question_famous_person'].nunique().reset_index()\
        .pivot_table(index='us_sample', columns='alive_dead', values='question_famous_person')

Number of bios


alive_dead,Alive,Dead
us_sample,,
Americans,171,598
Non-Americans,231,784


In [77]:
print("Number of bios")
nonrepeated_data[nonrepeated_data['current_answer']!=' '].groupby(['us_sample'])['question_famous_person'].nunique().reset_index()\
        .pivot_table(index='us_sample', values='question_famous_person')

Number of bios


,question_famous_person
us_sample,
Americans,803
Non-Americans,1056


In [78]:
print("Number of bios")
nonrepeated_data[nonrepeated_data['current_answer']!=' '].groupby(['alive_dead'])['question_famous_person'].nunique().reset_index()\
        .pivot_table(index='alive_dead', values='question_famous_person')

Number of bios


,question_famous_person
alive_dead,
Alive,254
Dead,842


In [79]:
nonrepeated_data.head()

,user_id,person_country,person_age,person_gender,question_id,q_id,question_famous_person,question_bplace_country,question_birthyear,question_deathyear,...,question_gender,question_date,question_time,current_answer,correct_answer_option,question_type,correct,country_comparison,alive_dead,us_sample
20998,6db20739-00f0-40cf-8b69-fbbd739358bb,NaN,NaN,NaN,56,1,Patrick Swayze,United States,1952.0,2009.0,...,M,2022-11-02,00:34:42.036,,c,medium,0,None,Dead,None
20718,6db20739-00f0-40cf-8b69-fbbd739358bb,NaN,NaN,NaN,59,2,Frank Sinatra,United States,1915.0,1998.0,...,M,2022-11-02,00:34:58.242,Frank Sinatra,c,hard,1,None,Dead,None
20791,7f3d5033-7a5d-4799-83e0-d3a344a9d841,United States,Others,Men,55,1,George Westinghouse,United States,1846.0,1914.0,...,M,2022-11-02,00:52:04.613,Inventor,a,medium,1,Same,Dead,Americans
20719,7f3d5033-7a5d-4799-83e0-d3a344a9d841,United States,Others,Men,59,2,Frank Sinatra,United States,1915.0,1998.0,...,M,2022-11-02,00:52:07.767,Frank Sinatra,b,hard,1,Same,Dead,Americans
20828,7f3d5033-7a5d-4799-83e0-d3a344a9d841,United States,Others,Men,52,3,Sylvester Stallone,United States,1946.0,NaN,...,M,2022-11-02,00:52:12.416,United States,b,easy,1,Same,Alive,Americans


In [80]:
nonrepeated_data.to_csv("clean_game.csv",index=False)

In [83]:
pd.read_csv("clean_game.csv")['user_id'].nunique()

3858

In [84]:
pd.read_csv("clean_game.csv")['question_id'].nunique()

1342

In [85]:
pd.read_csv("clean_game.csv")['question_famous_person'].nunique()

1144

In [1]:
import pandas as pd
game = pd.read_csv("clean_game.csv")

In [8]:
def clean_(text):
    return np.sort(np.unique([word.strip().replace("]", "").replace("[", "") for word in text.split(',') ]))
    
participants_file = "../../popularity_data/games/participant_202404121349.csv"
df = pd.read_csv(participants_file)
df.head()

,id,user_id,ip_hash,country_id,location_id,age_id,sex_id,language_ids,education_id,universe,locale,score_bot,created_at
0,10744,91cace8d-26c7-4ec9-99f5-dadae51833f8,5ff2427bab403b3a8d78a9a00c0dd21024d877ae8a4530...,840,46,1,2,"[4,4,1]",99,birthle,en,0.9,2024-04-12 12:59:35.867 +0100
1,10743,e833367d-9779-46c5-9d37-11d517ed53f7,327c14a3495875674d1e5e784781a7f9cdaeaece4fc475...,840,99,3,98,"[1,3]",4,trivia,en,0.7,2024-04-12 09:45:03.893 +0100
2,10742,00faed33-3ede-4c91-87e5-3bc4d4b98174,433a594cd132b73cf22102608af74f4ef1f56fa2d5dd72...,356,99,99,99,"[1,98,9,9,9]",99,trivia,en,0.7,2024-04-12 07:31:52.401 +0100
3,10741,265fc9ec-53ad-490e-b061-0b215526cefa,1423760b1d9718734eb6b3343511e7d437cc26e5b8fb11...,528,99,4,2,"[1,98]",4,trivia,en,0.9,2024-04-12 07:25:50.801 +0100
4,10740,47ec6d43-8ef6-4b34-983d-331cb7929718,e321317c77b431d78a1ae1e69d39dd0e60fd31b135c7a8...,840,43,2,2,"[1,3]",3,trivia,en,0.3,2024-04-12 06:37:20.089 +0100


In [9]:
df['created_at'].min()

'2022-11-16 19:52:36.547 +0000'

In [3]:
languages_code = {
    1: "english",
    2: "portuguese",
    3: "spanish",
    4: "italian",
    5: "french",
    6: "german",
    7: "chinese",
    8: "japanese",
    9: "hindi",
    10: "russian",
    11: "polish",
    12: "mandarin",
    13: "arabic",
    14: "bengali",
    15: "indonesian",
    16: "korean",
    98: "other",
    99: "skip",
}

In [4]:
import numpy as np
languages = df[['user_id','language_ids']].dropna().copy()
languages['language_ids'] = languages['language_ids'].apply(lambda x: clean_(x))
languages['person_number_languages'] = languages['language_ids'].apply(lambda x: len(x))
languages = languages[languages['person_number_languages']>0]
np.unique(np.hstack(languages['language_ids'].values))

array(['', '1', '10', '11', '12', '13', '14', '15', '16', '2', '3', '4',
       '5', '6', '7', '8', '9', '98', '99'], dtype='<U2')

In [5]:
languages['language_ids'].head().apply(lambda x: [languages_code[int(each)].capitalize() for each in list(x) if str(each) != ''])

0         [English, Italian]
1         [English, Spanish]
2    [English, Hindi, Other]
3           [English, Other]
4         [English, Spanish]
Name: language_ids, dtype: object

In [6]:
languages['person_language_labels'] = languages['language_ids'].apply(lambda x: [languages_code[int(each)].capitalize() for each in list(x) if str(each) != ''])
languages.head()

,user_id,language_ids,person_number_languages,person_language_labels
0,91cace8d-26c7-4ec9-99f5-dadae51833f8,"[1, 4]",2,"[English, Italian]"
1,e833367d-9779-46c5-9d37-11d517ed53f7,"[1, 3]",2,"[English, Spanish]"
2,00faed33-3ede-4c91-87e5-3bc4d4b98174,"[1, 9, 98]",3,"[English, Hindi, Other]"
3,265fc9ec-53ad-490e-b061-0b215526cefa,"[1, 98]",2,"[English, Other]"
4,47ec6d43-8ef6-4b34-983d-331cb7929718,"[1, 3]",2,"[English, Spanish]"


In [7]:
# languages['language_second'] = languages['language_ids'].apply(lambda x: clean_(x)[1] if len(clean_(x)) > 1 else '')
# languages['language_first'] = languages['language_first'].apply(lambda x: int(x) if str(x) != '' else None)
# languages['language_second'] = languages['language_second'].apply(lambda x: int(x) if str(x) != '' else None)

In [8]:
# languages.rename(columns= {"language_ids":"person_language_ids",
#                           "language_first":"person_country_game_firstlanguage",
#                           "language_second":"person_country_game_secondlanguage"}, inplace=True)

In [9]:
len(game)

41231

In [10]:
game = game.merge(languages[['user_id','person_language_labels','person_number_languages']], how='left')

In [11]:
official_languages = {
    "United States": ("English", "Spanish"),
    "France": ("French", "English"),
    "Germany": ("German", "English"),
    "Netherlands": ("Dutch", "English"),
    "United Kingdom": ("English", "Polish"),
    "Italy": ("Italian", "English"),
    "Poland": ("Polish", "English"),
    "Russia": ("Russian", "Tatar"),
    "Spain": ("Spanish", "Catalan"),
    "Austria": ("German", "English"),
    "Japan": ("Japanese", "English"),
    "Ukraine": ("Ukrainian", "Russian"),
    "Ireland": ("English", "Irish"),
    "Serbia": ("Serbian", "Hungarian"),
    "Georgia": ("Georgian", "Russian"),
    "Canada": ("English", "French"),
    "Belgium": ("Dutch", "French"),
    "Estonia": ("Estonian", "Russian"),
    "Czechia": ("Czech", "Slovak"),
    "Egypt": ("Arabic", "English"),
    "Denmark": ("Danish", "English"),
    "Romania": ("Romanian", "Hungarian"),
    "Iran": ("Persian (Farsi)", "Azerbaijani"),
    "Croatia": ("Croatian", "Serbian"),
    "Argentina": ("Spanish", "Italian"),
    "Hungary": ("Hungarian", "German"),
    "Brazil": ("Portuguese", "Spanish"),
    "Slovakia": ("Slovak", "Hungarian"),
    "Lithuania": ("Lithuanian", "Russian"),
    "Australia": ("English", "Mandarin"),
    "Slovenia": ("Slovene", "Serbian"),
    "Norway": ("Norwegian", "English"),
    "Belarus": ("Belarusian", "Russian"),
    "India": ("Hindi", "English"),
    "Portugal": ("Portuguese", "Mirandese"),
    "Bangladesh": ("Bengali", "Chakma"),
    "Finland": ("Finnish", "Swedish"),
    "Switzerland": ("German", "French"),
    "Turkey": ("Turkish", "Kurdish"),
    "Hong Kong": ("Cantonese", "English"),
    "China": ("Mandarin", "Cantonese"),
    "Uruguay": ("Spanish", "Portuguese"),
    "Bulgaria": ("Bulgarian", "Turkish"),
    "Sweden": ("Swedish", "English"),
    "South Korea": ("Korean", "English"),
    "South Africa": ("Zulu", "Xhosa"),
    "Bosnia and Herzegovina": ("Bosnian", "Serbian"),
    "Indonesia": ("Indonesian", "Javanese"),
    "Peru": ("Spanish", "Quechua")
}


country_languages = {
    "United States": ("English", "Spanish"),
    "France": ("French", "English", "Spanish"),
    "Germany": ("German", "English"),
    "Netherlands": ("Dutch", "English"),
    "United Kingdom": ("English", "Scotish"),
    "Italy": ("Italian", "English"),
    "Poland": ("Polish", "English"),
    "Russia": ("Russian", "Belarusian", "Ukrainian"),
    "Spain": ("Spanish", "Catalan"),
    "Austria": ("German", "English"),
    "Japan": ("Japanese", "English"),
    "Ukraine": ("Ukrainian", "Russian"),
    "Ireland": ("English", "Irish"),
    "Serbia": ("Serbian", "Hungarian"),
    "Georgia": ("Georgian", "Russian"),
    "Canada": ("English", "French"),
    "Belgium": ("Dutch", "French"),
    "Estonia": ("Estonian", "Russian"),
    "Czechia": ("Czech", "Slovak"),
    "Egypt": ("Arabic", "English"),
    "Denmark": ("Danish", "English"),
    "Romania": ("Romanian", "Hungarian"),
    "Iran": ("Persian (Farsi)", "Azerbaijani"),
    "Croatia": ("Croatian", "Serbian"),
    "Argentina": ("Spanish", "Italian"),
    "Hungary": ("Hungarian", "German"),
    "Brazil": ("Portuguese", "Spanish"),
    "Slovakia": ("Slovak", "Hungarian"),
    "Lithuania": ("Lithuanian", "Russian"),
    "Australia": ("English", "Mandarin"),
    "Slovenia": ("Slovene", "Serbian"),
    "Norway": ("Norwegian", "English"),
    "Belarus": ("Belarusian", "Russian"),
    "India": ("Hindi", "English"),
    "Portugal": ("Portuguese", "Mirandese"),
    "Bangladesh": ("Bengali", "Chakma"),
    "Finland": ("Finnish", "Swedish"),
    "Switzerland": ("German", "French"),
    "Turkey": ("Turkish", "Kurdish"),
    "Hong Kong": ("Cantonese", "English"),
    "China": ("Mandarin", "Cantonese"),
    "Uruguay": ("Spanish", "Portuguese"),
    "Bulgaria": ("Bulgarian", "Turkish"),
    "Sweden": ("Swedish", "English"),
    "South Korea": ("Korean", "English"),
    "South Africa": ("Zulu", "Xhosa"),
    "Bosnia and Herzegovina": ("Bosnian", "Serbian"),
    "Indonesia": ("Indonesian", "Javanese"),
    "Peru": ("Spanish", "Quechua"),
    "Chile": ("Spanish","None"),
    "New Zealand": ("Māori","English"),
    "Israel": ()
}


In [1]:
import numpy as np
import pandas as pd
# pd.DataFrame(np.unique(np.hstack([list(official_languages.keys()), list(country_languages.keys())])))

In [2]:
languages_df = pd.read_csv("data/LanguageDistribution.csv")
languages_df = languages_df[languages_df['Percentage of People']>=25]
languages_df.head()

,Country,Language,Percentage of People
0,Angola,Portuguese,75.0
1,Argentina,Spanish,95.2
2,Australia,English,72.0
4,Austria,German,88.6
5,Bangladesh,Bangla,98.8


In [8]:
languages_df[languages_df['Percentage of People']<26]

,Country,Language,Percentage of People
38,Hungary,English,25.3


In [11]:
def get_language(country, order_id=1):
    langs = languages_df[(languages_df["Country"]==country)].sort_values(by='Percentage of People', ascending=False)
    if len(langs) >= order_id:
        return langs['Language'].values[order_id-1]
    return None

get_language('Hungary', order_id=1), get_language('Hungary', order_id=2), get_language('Austria', order_id=3)

('Hungarian', 'English', None)

In [15]:
game["question_country_firstlanguage"] = game["question_bplace_country"].apply(lambda x: get_language(x, order_id=1) if str(x)!='nan' else None)
game["question_country_secondlanguage"] = game["question_bplace_country"].apply(lambda x: get_language(x, order_id=2) if str(x)!='nan' else None)

In [16]:
game["person_country_inferred_firstlanguage"] = game["person_country"].apply(lambda x: get_language(x, order_id=1) if str(x)!='nan' else None)
game["person_country_inferred_secondlanguage"] = game["person_country"].apply(lambda x: get_language(x, order_id=2) if str(x)!='nan' else None)

In [17]:
# game["question_country_firstlanguage"] = game["question_bplace_country"].apply(lambda x: country_languages[x][0].capitalize() if str(x)!='nan' else None)
# game["question_country_secondlanguage"] = game["question_bplace_country"].apply(lambda x: country_languages[x][1].capitalize() if str(x)!='nan' else None)

In [18]:
# game["person_country_inferreffirstlanguage"] = game["person_country"].apply(lambda x: country_languages[x][0].capitalize() if str(x)!='nan' else None)
# game["person_country_inferredsecondlanguage"] = game["person_country"].apply(lambda x: country_languages[x][1].capitalize() if str(x)!='nan' else None)

In [19]:
# import ast
# game["person_language_ids"] = game["person_language_ids"].apply(lambda x: ast.literal_eval(x) if str(x)!='nan' else None)
# game.head()

In [20]:
# game["person_number_languages"] = game["person_language_ids"].apply(lambda x: len(np.unique(x)) if x!=None else 0)

In [21]:
aux = game[['person_country','person_country_inferred_firstlanguage']].drop_duplicates()
aux[(aux['person_country'].notna()) & (aux['person_country_inferred_firstlanguage'].isna())].to_csv("aux.csv")

In [22]:
aux[(aux['person_country'].notna()) & (aux['person_country_inferred_firstlanguage'].isna())]

,person_country,person_country_inferred_firstlanguage
2955,Others,None
3919,Antarctica,None
5100,Antigua and Barbuda,None
5757,South Africa,None
5799,Kenya,None
6130,Czech Republic,None
6322,United Arab Emirates,None


In [23]:
game.to_csv("data/clean_game_with_languages.csv", index=False)

In [24]:
game.head()

,user_id,person_country,person_age,person_gender,question_id,q_id,question_famous_person,question_bplace_country,question_birthyear,question_deathyear,...,correct,country_comparison,alive_dead,us_sample,person_language_labels,person_number_languages,question_country_firstlanguage,question_country_secondlanguage,person_country_inferred_firstlanguage,person_country_inferred_secondlanguage
0,6db20739-00f0-40cf-8b69-fbbd739358bb,NaN,NaN,NaN,56,1,Patrick Swayze,United States,1952.0,2009.0,...,0,NaN,Dead,NaN,NaN,NaN,English,None,None,None
1,6db20739-00f0-40cf-8b69-fbbd739358bb,NaN,NaN,NaN,59,2,Frank Sinatra,United States,1915.0,1998.0,...,1,NaN,Dead,NaN,NaN,NaN,English,None,None,None
2,7f3d5033-7a5d-4799-83e0-d3a344a9d841,United States,Others,Men,55,1,George Westinghouse,United States,1846.0,1914.0,...,1,Same,Dead,Americans,"[English, Bengali]",2.0,English,None,English,None
3,7f3d5033-7a5d-4799-83e0-d3a344a9d841,United States,Others,Men,59,2,Frank Sinatra,United States,1915.0,1998.0,...,1,Same,Dead,Americans,"[English, Bengali]",2.0,English,None,English,None
4,7f3d5033-7a5d-4799-83e0-d3a344a9d841,United States,Others,Men,52,3,Sylvester Stallone,United States,1946.0,NaN,...,1,Same,Alive,Americans,"[English, Bengali]",2.0,English,None,English,None
